## Contextual (local) click embeddings using Word2Vec

In [1]:
from utils.ml import tokenize, load_data, data_dir
from pyspark.ml.feature import Word2Vec

df = load_data(data_dir)
sequences, tokens = tokenize(df)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/23 06:48:29 WARN Utils: Your hostname, DESKTOP-UQF5BSK, resolves to a loopback address: 127.0.1.1; using 172.24.225.163 instead (on interface eth0)
26/04/23 06:48:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 06:48:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


TOKENIZED DF:


+----+-----+---+-----+-------+----------+--------------------+
|year|month|day|order|country|session ID|            sequence|
+----+-----+---+-----+-------+----------+--------------------+
|2008|    4|  1|    9|     29|         1|[197, 149, 51, 91...|
|2008|    4|  1|   10|     29|         2|[87, 12, 215, 9, ...|
+----+-----+---+-----+-------+----------+--------------------+
only showing top 2 rows
TOKEN INFO: 
+----------------------+-----------------------+------+--------+-----------------+-----+-------+----+---+
|page 1 (main category)|page 2 (clothing model)|colour|location|model photography|price|price 2|page| id|
+----------------------+-----------------------+------+--------+-----------------+-----+-------+----+---+
|                     4|                    P51|     9|       5|                1|   28|      2|   3|  0|
|                     4|                    P57|     4|       1|                1|   43|      1|   4|  1|
+----------------------+-----------------------+------+

### Word2Vec

In [2]:
w2v = Word2Vec(vectorSize=32, inputCol="sequence", maxSentenceLength = 195).fit(sequences)
vectors = w2v.getVectors()
vectors.show(5)

+----+--------------------+
|word|              vector|
+----+--------------------+
|  98|[-0.1281019002199...|
|  67|[-0.3852678835391...|
| 169|[0.12478501349687...|
| 120|[-0.1710163801908...|
| 193|[-0.0860464125871...|
+----+--------------------+
only showing top 5 rows


In [3]:
from pyspark.sql import functions as F
melted_sequences = sequences.withColumn("id", F.explode("sequence")).drop("order")
melted_sequences = melted_sequences.join(vectors.withColumnRenamed("word", "id"), on = "id", how = "left").drop("sequence")
melted_sequences.show(5)

+---+----+-----+---+-------+----------+--------------------+
| id|year|month|day|country|session ID|              vector|
+---+----+-----+---+-------+----------+--------------------+
|197|2008|    4|  1|     29|         1|[0.25304126739501...|
|149|2008|    4|  1|     29|         1|[0.45443812012672...|
| 51|2008|    4|  1|     29|         1|[-0.4271827638149...|
| 91|2008|    4|  1|     29|         1|[-0.2043882757425...|
|111|2008|    4|  1|     29|         1|[-0.3095721602439...|
+---+----+-----+---+-------+----------+--------------------+
only showing top 5 rows


### Let's try to cluster mean pooling

In [6]:
from pyspark.ml.stat import Summarizer
from pyspark.sql import Window
from pyspark.ml.feature import Normalizer

w = Window.partitionBy("session ID")
mean = Summarizer.metrics("mean")
pooled = melted_sequences.select("Session ID", mean.summary(F.col("vector")).over(w).alias("context"))
pooled = pooled.drop_duplicates(subset=["session ID"]).withColumn("context", F.col("context.mean"))
normalized = Normalizer(inputCol="context", outputCol="context_norm").transform(pooled)